# 2.6 — Conjugate gradient

Conjugate gradient solves quadratic systems by choosing directions that do not reintroduce errors already removed.

CG solves SPD quadratic systems using residuals, inner products, and matrix-vector products. It is a second-order workhorse when a dense inverse is too expensive. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

SEED = 20260701
rng = np.random.default_rng(SEED)


def quadratic_surface(name, H, center, x0, project=None):
    H = np.asarray(H, dtype=float)
    center = np.asarray(center, dtype=float)
    x0 = np.asarray(x0, dtype=float)

    def loss(x):
        z = np.asarray(x, dtype=float) - center
        return 0.5 * float(z @ H @ z)

    def grad(x):
        z = np.asarray(x, dtype=float) - center
        return H @ z

    def hess(x):
        return H

    return {
        "name": name,
        "kind": "quadratic",
        "dim": len(x0),
        "x0": x0,
        "loss": loss,
        "grad": grad,
        "hess": hess,
        "project": project,
        "center": center,
    }


def rosenbrock_surface():
    def loss(x):
        x = np.asarray(x, dtype=float)
        a = x[0]
        b = x[1]
        return float(100.0 * (b - a * a) ** 2 + (1.0 - a) ** 2)

    def grad(x):
        x = np.asarray(x, dtype=float)
        a = x[0]
        b = x[1]
        return np.array([
            -400.0 * a * (b - a * a) - 2.0 * (1.0 - a),
            200.0 * (b - a * a),
        ])

    def hess(x):
        x = np.asarray(x, dtype=float)
        a = x[0]
        b = x[1]
        return np.array([
            [1200.0 * a * a - 400.0 * b + 2.0, -400.0 * a],
            [-400.0 * a, 200.0],
        ])

    return {
        "name": "D3 nonconvex Rosenbrock valley",
        "kind": "rosenbrock",
        "dim": 2,
        "x0": np.array([-1.2, 1.0]),
        "loss": loss,
        "grad": grad,
        "hess": hess,
        "project": None,
        "center": np.array([1.0, 1.0]),
    }


def logistic_surface():
    data = load_breast_cancer()
    X = StandardScaler().fit_transform(data.data)
    X = np.column_stack([np.ones(X.shape[0]), X])
    y = data.target.astype(float)
    reg = 0.05

    def loss(w):
        z = X @ w
        yz = y * z
        logistic = np.logaddexp(0.0, z) - yz
        penalty = 0.5 * reg * float(w[1:] @ w[1:])
        return float(np.mean(logistic) + penalty)

    def grad(w):
        z = X @ w
        p = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
        g = X.T @ (p - y) / X.shape[0]
        g[1:] = g[1:] + reg * w[1:]
        return g

    def hess(w):
        z = X @ w
        p = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
        weights = p * (1.0 - p)
        Xw = X * weights[:, None]
        H = X.T @ Xw / X.shape[0]
        H[1:, 1:] = H[1:, 1:] + reg * np.eye(X.shape[1] - 1)
        return H

    return {
        "name": "D4 breast-cancer logistic loss",
        "kind": "logistic",
        "dim": X.shape[1],
        "x0": np.zeros(X.shape[1]),
        "loss": loss,
        "grad": grad,
        "hess": hess,
        "project": None,
        "center": np.zeros(X.shape[1]),
        "X_shape": X.shape,
        "classes": sorted(set(data.target.tolist())),
    }


def make_high_dim_box_surface(dim=20):
    Q, _ = np.linalg.qr(rng.normal(size=(dim, dim)))
    eigs = np.geomspace(1.0, 120.0, dim)
    H = Q @ np.diag(eigs) @ Q.T
    center = np.linspace(-0.8, 0.8, dim)
    x0 = np.linspace(1.4, -1.4, dim)

    def project(x):
        return np.clip(x, -1.0, 1.0)

    return quadratic_surface("D5 high-dimensional box-constrained SPD bowl", H, center, x0, project=project)


def make_loss_ladder():
    d1 = quadratic_surface(
        "D1 2-D quadratic bowl",
        np.diag([2.0, 4.0]),
        np.array([1.0, -1.0]),
        np.array([-2.0, 2.0]),
    )
    d2 = quadratic_surface(
        "D2 anisotropic ill-conditioned quadratic",
        np.diag([1.0, 80.0]),
        np.array([-1.0, 1.0]),
        np.array([2.5, -2.0]),
    )
    return [d1, d2, rosenbrock_surface(), logistic_surface(), make_high_dim_box_surface()]


def project_if_needed(surface, x):
    if surface.get("project") is None:
        return np.asarray(x, dtype=float)
    return surface["project"](np.asarray(x, dtype=float))


def run_optimizer(surface, steps=120, eta=0.05, tol=1e-6):
    x = project_if_needed(surface, surface["x0"])
    losses = []
    path = [x.copy()]
    for step in range(steps):
        loss_value = surface["loss"](x)
        losses.append(loss_value)
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        trial = x - eta * g
        x = project_if_needed(surface, trial)
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def backtracking_line_search(loss, grad, x, direction, alpha0=1.0, rho=0.5, c=0.5):
    alpha = alpha0
    f0 = loss(x)
    g0 = grad(x)
    slope = float(g0 @ direction)
    while loss(x + alpha * direction) > f0 + c * alpha * slope:
        alpha = rho * alpha
        if alpha < 1e-10:
            break
    return alpha


def line_search_optimizer(surface, steps=80, tol=1e-6):
    x = project_if_needed(surface, surface["x0"])
    losses = []
    path = [x.copy()]
    for step in range(steps):
        losses.append(surface["loss"](x))
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        direction = -g
        alpha = backtracking_line_search(surface["loss"], surface["grad"], x, direction)
        x = project_if_needed(surface, x + alpha * direction)
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def bfgs_update(B, s, y):
    Bs = B @ s
    sBs = float(s @ Bs)
    ys = float(y @ s)
    if ys <= 1e-12 or sBs <= 1e-12:
        return B
    return B - np.outer(Bs, Bs) / sBs + np.outer(y, y) / ys


def damped_newton_optimizer(surface, steps=50, tol=1e-6, damping=1e-4):
    x = project_if_needed(surface, surface["x0"])
    losses = []
    path = [x.copy()]
    for step in range(steps):
        losses.append(surface["loss"](x))
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        H = surface["hess"](x)
        shift = damping
        eye = np.eye(len(x))
        while np.min(np.linalg.eigvalsh(H + shift * eye)) <= 0.0:
            shift = 10.0 * shift
        direction = -np.linalg.solve(H + shift * eye, g)
        if float(direction @ g) >= 0.0:
            direction = -g
        alpha = backtracking_line_search(surface["loss"], surface["grad"], x, direction, c=1e-4)
        x = project_if_needed(surface, x + alpha * direction)
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def bfgs_optimizer(surface, steps=60, tol=1e-6):
    x = project_if_needed(surface, surface["x0"])
    dim = len(x)
    B = np.eye(dim)
    losses = []
    path = [x.copy()]
    for step in range(steps):
        losses.append(surface["loss"](x))
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        direction = -np.linalg.solve(B + 1e-8 * np.eye(dim), g)
        if float(direction @ g) >= 0.0:
            direction = -g
        alpha = backtracking_line_search(surface["loss"], surface["grad"], x, direction, c=1e-4)
        x_next = project_if_needed(surface, x + alpha * direction)
        s = x_next - x
        y = surface["grad"](x_next) - g
        B = bfgs_update(B, s, y)
        x = x_next
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def conjugate_gradient(A, b, x0=None, tol=1e-8, max_iter=None):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    if x0 is None:
        x = np.zeros_like(b)
    else:
        x = np.asarray(x0, dtype=float).copy()
    if max_iter is None:
        max_iter = len(b)
    r = b - A @ x
    p = r.copy()
    rs_old = float(r @ r)
    residuals = [np.sqrt(rs_old)]
    path = [x.copy()]
    for k in range(max_iter):
        Ap = A @ p
        denom = float(p @ Ap)
        if denom <= 0.0:
            raise ValueError("CG requires symmetric positive-definite curvature")
        alpha = rs_old / denom
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = float(r @ r)
        residuals.append(np.sqrt(rs_new))
        path.append(x.copy())
        if np.sqrt(rs_new) < tol:
            break
        beta = rs_new / rs_old
        p = r + beta * p
        rs_old = rs_new
    return {
        "x": x,
        "residuals": np.array(residuals),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def make_cg_systems():
    systems = []
    base = make_loss_ladder()
    for surface in base:
        if surface["kind"] == "rosenbrock":
            point = np.array([1.0, 1.0])
            A = surface["hess"](point) + 1e-3 * np.eye(2)
            b = A @ point
            name = "D3 Rosenbrock SPD local Newton system"
        elif surface["kind"] == "logistic":
            point = surface["x0"]
            A = surface["hess"](point)
            b = -surface["grad"](point)
            name = "D4 logistic Hessian-vector SPD system"
        else:
            A = surface["hess"](surface["x0"])
            b = A @ surface["center"]
            name = surface["name"].replace("loss", "system")
        systems.append({
            "name": name,
            "A": A,
            "b": b,
            "x0": np.zeros_like(b),
        })
    return systems


def preview_ladder(ladder):
    for idx, item in enumerate(ladder, start=1):
        if "A" in item:
            shape = item["A"].shape
            extra = f"system_shape={shape}"
            sample = item["b"][: min(3, len(item["b"]))]
        else:
            shape = (item["dim"],)
            extra = f"dim={item['dim']} kind={item['kind']}"
            sample = item["x0"][: min(3, len(item["x0"]))]
        print(f"D{idx}: {item['name']} | {extra} | sample={sample}")


def plot_loss_contours(ax, surface, path, title):
    path = np.asarray(path)
    if path.ndim == 1:
        path = path[:, None]
    if path.shape[1] == 1:
        xs = np.linspace(-3.0, 3.0, 120)
        ys = np.zeros_like(xs)
        vals = np.array([surface["loss"](np.array([x])) for x in xs])
        ax.plot(xs, vals)
        ax.set_title(title)
        return
    x_min = min(np.min(path[:, 0]) - 0.5, -2.5)
    x_max = max(np.max(path[:, 0]) + 0.5, 2.5)
    y_min = min(np.min(path[:, 1]) - 0.5, -2.5)
    y_max = max(np.max(path[:, 1]) + 0.5, 2.5)
    xs = np.linspace(x_min, x_max, 100)
    ys = np.linspace(y_min, y_max, 100)
    xx, yy = np.meshgrid(xs, ys)
    base = path[-1].copy()
    zz = np.zeros_like(xx)
    for i in range(xx.shape[0]):
        for j in range(xx.shape[1]):
            point = base.copy()
            point[0] = xx[i, j]
            point[1] = yy[i, j]
            zz[i, j] = surface["loss"](point)
    ax.contour(xx, yy, zz, levels=20)
    ax.plot(path[:, 0], path[:, 1], marker="o", markersize=2)
    ax.set_title(title)


def plot_system_contours(ax, system, path, title):
    A = system["A"]
    b = system["b"]
    path = np.asarray(path)
    if path.shape[1] < 2:
        ax.plot(np.arange(len(path)), path[:, 0])
        ax.set_title(title)
        return
    x_min = min(np.min(path[:, 0]) - 0.5, -2.0)
    x_max = max(np.max(path[:, 0]) + 0.5, 2.0)
    y_min = min(np.min(path[:, 1]) - 0.5, -2.0)
    y_max = max(np.max(path[:, 1]) + 0.5, 2.0)
    xs = np.linspace(x_min, x_max, 90)
    ys = np.linspace(y_min, y_max, 90)
    xx, yy = np.meshgrid(xs, ys)
    base = path[-1].copy()
    zz = np.zeros_like(xx)
    for i in range(xx.shape[0]):
        for j in range(xx.shape[1]):
            point = base.copy()
            point[0] = xx[i, j]
            point[1] = yy[i, j]
            zz[i, j] = 0.5 * float(point @ A @ point) - float(b @ point)
    ax.contour(xx, yy, zz, levels=20)
    ax.plot(path[:, 0], path[:, 1], marker="o", markersize=2)
    ax.set_title(title)

## The concept, built once (D1)

For SPD $A$, CG uses $\alpha_k=\frac{r_k^Tr_k}{p_k^TAp_k}$ and $\beta_k=\frac{r_{k+1}^Tr_{k+1}}{r_k^Tr_k}$ with $r=b-Ax$.

The D1 check is intentionally tiny: it plugs in the lesson's exact numbers before the same optimizer machinery is used on harder rungs.

In [ ]:
A = np.array([[4.0, 1.0], [1.0, 3.0]])
b = np.array([1.0, 2.0])
x0 = np.array([0.0, 0.0])
r0 = b - A @ x0
p0 = r0.copy()
denominator = float(p0 @ A @ p0)
alpha0 = float(r0 @ r0) / denominator
assert np.allclose(r0, np.array([1.0, 2.0]))
assert np.isclose(denominator, 20.0)
assert np.isclose(alpha0, 0.25)
print("r0:", r0)
print("alpha0:", alpha0)

After the first step, the new residual determines $\beta$. The updated direction is $A$-conjugate to the old one, so it does not undo progress already made.

In [ ]:
x1 = x0 + alpha0 * p0
r1 = r0 - alpha0 * (A @ p0)
beta = float(r1 @ r1) / float(r0 @ r0)
p1 = r1 + beta * p0
conjugacy = float(p0 @ A @ p1)
assert np.allclose(r1, np.array([-0.5, 0.25]))
assert np.isclose(beta, 0.0625)
assert np.isclose(conjugacy, 0.0, atol=1e-10)
print("r1:", r1)
print("beta:", beta)
print("p0^T A p1:", conjugacy)

## Dataset ladder: D1→D5 CG systems

Inline F4 ladder, no shared helper import: D1 quadratic bowl, D2 ill-conditioned quadratic, D3 Rosenbrock/nonconvex, D4 real `load_breast_cancer` logistic loss, and D5 high-dimensional or constrained. The metric is iterations to tolerance.

In [ ]:
ladder = make_cg_systems()
preview_ladder(ladder)

## Run the same method across D1→D5

Only the method for this lesson changes; the ladder stays fixed. Collect one metric per rung: iterations to tolerance.

In [ ]:
ladder = make_cg_systems()
results = []
for rung, system in enumerate(ladder, start=1):
    max_iter = min(120, len(system["b"]))
    result = conjugate_gradient(system["A"], system["b"], system["x0"], tol=1e-7, max_iter=max_iter)
    results.append(result)
    final_residual = result["residuals"][-1]
    print(f"D{rung} | iterations_to_tol={result['iterations']} | residual={final_residual:.6e}")

## Results visualization

The closing figure has two parts: optimizer trajectories on contour panels, then the metric curve across D1→D5.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for idx, (system, result) in enumerate(zip(ladder, results)):
    plot_system_contours(axes[idx], system, result["path"], f"D{idx + 1}")
plt.tight_layout()
plt.show()

metrics = [result["iterations"] for result in results]
plt.figure(figsize=(6, 3.5))
plt.plot(range(1, 6), metrics, marker="o")
plt.xlabel("rung")
plt.ylabel("iterations to tolerance")
plt.title("CG iterations across the ladder")
plt.grid(True)
plt.show()

## Pitfall on the hardest rung

Pitfall on D5: CG requires SPD curvature. Non-SPD or extremely ill-conditioned systems can create nonpositive denominators or lose conjugacy; project to SPD and use preconditioning or restarts.

In [ ]:
A_bad = np.array([[1.0, 0.0], [0.0, -0.1]])
b_bad = np.array([1.0, 1.0])
try:
    bad_result = conjugate_gradient(A_bad, b_bad, max_iter=2)
    print("bad residual:", bad_result["residuals"][-1])
except ValueError as err:
    print("non-SPD failure:", err)

w, V = np.linalg.eigh(A_bad)
w_fixed = np.maximum(w, 1e-3)
A_fixed = V @ np.diag(w_fixed) @ V.T
fixed_result = conjugate_gradient(A_fixed, b_bad, max_iter=10)
print("fixed SPD eigenvalues:", np.linalg.eigvalsh(A_fixed))
print("fixed residual:", fixed_result["residuals"][-1])

## Evaluate it + Practice

- Metric: iterations to tolerance; compare against a no-skill baseline that keeps the initial point or takes a fixed tiny step.
- Sanity check: on D1, verify the output against the closed-form minimizer or exact linear-system solution.
- Ablation: turn off the key safeguard or curvature idea and confirm the metric worsens.
- Failure signals: nondecreasing loss, nonpositive CG denominator, exploding path length, or a tiny gradient with bad curvature.
- Reproducibility: keep the seed fixed and do not download data; `load_breast_cancer` is bundled with sklearn.

Practice prompts:

1. Change one tolerance and predict which rung changes the most.

2. Replace D2's condition number and compare the metric curve.

3. Add one diagnostic print that would catch the pitfall earlier.